In [3]:
import torch
from torch import nn

In [7]:
from torch.utils.tensorboard import SummaryWriter

In [10]:
writer = SummaryWriter()

In [11]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Use all available CPU cores for data loading by default.
NUM_WORKERS = os.cpu_count() or 0

def create_dataloader(train_dir: str,
                      test_dir: str,
                      transforms: transforms.Compose,
                      batch_size: int,
                      num_worksers: int = NUM_WORKERS):
    # Load training images from folder structure.
    train_data  = datasets.ImageFolder(root=train_dir,
                                       transform=transforms)

    # Load test images from folder structure.
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)

    # Get class names from training dataset folders.
    class_names = train_data.classes

    # Create DataLoader for training data (shuffled each epoch).
    train_dataloader = DataLoader(dataset=train_data,
                                  shuffle=True,
                                  batch_size=batch_size,
                                  num_workers=num_worksers)

    # Create DataLoader for test data (no shuffle for evaluation).
    test_dataloader =  DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=False,
                                  num_workers=num_worksers)

    # Return both dataloaders and class labels.
    return train_dataloader, test_dataloader, class_names


In [12]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape:int, hidden_unit:int, output_shape:int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.dense_layer = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=hidden_unit *16*16, out_features=output_shape)
        )
    def forward(self,x):
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = self.dense_layer(out)
        return out

In [13]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
              ):
    model.train() # Modelimizi train moduna alıyoruz.
    
    train_loss = 0 # Train Loss değerlerini tutmak için bir değişken oluşturuyoruz.
    train_acc = 0 # Train Accuracy değerlerini tutmak için bir değişken oluşturuyoruz.

    for batch, (X,y) in enumerate(dataloader): # Batch size gerekli değil burada.
        y_pred = model(X) # Modelimize bir tahminde bulunduruyoruz.
        
        loss = loss_fn(y_pred,y) # Loss değerlerimizi loss_fn ile hesaplıyoruz.
        train_loss += loss.item() # Çıkan loss değerlerini train_loss değişlenine toplayarak atıyoruz.

        # Modelimizi backpropagation yapıyoruz.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Softmax kullanarak modelimize label tahmininde bulunduruyoruz.
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1) 
        
        train_acc += (y_pred_class == y).sum().item() / len(y_pred) # Accuracy değerlerimizi bir değişkende tutuyoruz.

    train_loss /= len(dataloader) # Train Loss değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz.
    train_acc /= len(dataloader) # Train Acc değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz
    return train_loss, train_acc # Geriye train_loss ve train_acc döndürüyoruz.

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
              ):
    model.eval() # Modelimizi test moduna alıyoruz.
    
    test_loss = 0 # test loss'ları tutmak için bir değişken oluşturuyoruz.
    test_acc = 0 # test accuracy'ları tutmak için bir değişken oluşturuyoruz.
    
    with torch.inference_mode(): # inference mode'a aldık.
        for batch, (X,y) in enumerate(dataloader): # batch gerekli değil fakat yine de aldık.
            test_pred = model(X) # modelimize tahmin ettiriyoruz.
            
            loss = loss_fn(test_pred,y) # loss'umuzu loss_fn ile hesaplıyoruz.
            test_loss += loss.item() # loss değerlerimizi test_loss değişkeninde topluyoruz.

            # Softmax activation function ile label tahmin ettiriyoruz.
            test_pred_label = torch.softmax(test_pred,dim=1).argmax(dim=1) 
            
            acc = (test_pred_label == y).sum().item() / len(test_pred) # Calculate accuracy
            test_acc += acc # Accuracy değerlerimizi toplayıp test_acc değişkenine atıyoruz.
            
    test_loss /= len(dataloader) # Test loss değerlerimizi dataloader boyuna bölüyoruz.
    test_acc /= len(dataloader) # Test acc değerlerimizi dataloader boyuna bölüyoruz.
    
    return test_loss, test_acc # Geriye test_loss ve test_acc döndürüyoruz.

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs:int = 10,
               experiment_name:str = "experiment"
              ):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                          )
        test_loss, test_acc = test_step(model = model,
                                           dataloader = test_dataloader,
                                           loss_fn = loss_fn,
                                          )
        print(f""" 
        Epoch:{epoch}
        Train Loss : {train_loss:.2f} -  Train Accuracy : {train_acc*100:.2f}
        Test Loss  : { test_loss:.2f} -  Test Accuracy  : {test_acc*100:.2f}
        """)
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

        writer = SummaryWriter(log_dir=f"runs/{experiment_name}")
        writer.add_scalars(
            main_tag="Loss",
            tag_scaler_dict={
                "train_loss":train_loss
                "test_loss": test_loss},
            global_step=epoch)

        writer.add_scalars(
            main_tag="Accuracy",
            tag_scaler_dict={
                "train_acc":train_acc
                "test_acc": test_acc},
            global_step=epoch)
        
        writer.add_graph(model=model,input_to_model=torch.randn(32,3,64,64))

    
        
    return results